In [1]:
from dataclasses import dataclass
import numpy as np
import os
import random
import torch
from torch import optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

if not os.path.exists("./KuiSCIMA"):
    !git clone https://github.com/SuziAI/KuiSCIMA.git
if not os.path.exists("./gui-tools"):
    !git clone https://github.com/SuziAI/gui-tools.git

from data_extraction import *
from image_manipulation import *
from model import *
from suzipu import *

BATCH_SIZE = 100

In [2]:
# Extract data from KuiSCIMA dataset
extract_data()

# Set seeds
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

In [3]:
total_dataset = open_suzipu_dataset("./TotalDataset/Music/")
total_dataset = get_cropped_dataset(total_dataset)

## Normalizes the dataset to have mean 0 and standard deviation of 1
#mean_list = torch.Tensor([torch.Tensor(entry["image"]).mean() for entry in total_dataset])/255
#mean = mean_list.mean()
#std = mean_list.std()

def normalize():
    return transforms.Normalize(mean=0.7102, std=0.0914)

def get_train_transforms(image_size=48):
    target_shrink = image_size - 8
    target_paste = image_size
    train_transforms = transforms.Compose([
        erode(percentage=0.2), # causes the notation to be thicker
        dilate(percentage=0.2), # causes the notation to be thinner
        transforms.ToTensor(), # convert from uint8 to [0, 1]
        shrink(is_random=True, target_size=target_shrink),
        paste_to_square(is_random=False, target_size=target_paste),
        salt_and_pepper(percentage=0.8), # introduces salt-and-pepper-noise
        lambda img: transforms.functional.invert(img), # inverts image, needed for rotations
        transforms.RandomRotation(degrees=(-9, 9)), # randomly rotate between -12 and 12 degrees
        #lambda img: transforms.functional.invert(img), # inverts image to original color scheme
        normalize(), # normalize mean and variance
    ])
    return train_transforms


def get_validation_transforms(image_size=48):
    target_shrink = image_size - 8
    target_paste = image_size
    validation_transforms = transforms.Compose([
        transforms.ToTensor(),
        shrink(is_random=False, target_size=target_shrink),
        paste_to_square(is_random=False, target_size=target_paste),
        lambda img: transforms.functional.invert(img), # inverts image, needed for rotations
        normalize(), # normalize mean and variance
    ])
    return validation_transforms


def get_test_transforms(image_size=48):
    return get_validation_transforms(image_size=image_size)

transformations = {
    "train": get_test_transforms,
    "validation": get_test_transforms,
    "test": get_test_transforms
}

# Temperature Scaling

In [4]:


def predict_data_loader(model, data_loader, scaled=True):
    model.eval()  
    predictions = []
    labels = []  
    with torch.no_grad():  
        for batch in data_loader:
            images = batch[0]
            
            if scaled:
                probs = model.forward(images)
            else:
                probs = model.forward_unscaled(images)
                
            predictions.extend(probs.cpu().numpy())
            labels.extend(batch[1].cpu().numpy())  
    return torch.from_numpy(np.array(predictions)), torch.from_numpy(np.array(labels))

def reliability_diagram(output, true_labels, n_bins=10, name="", plot=True):
    bins = torch.linspace(0, 1, n_bins + 1)
    bin_lowers = bins[:-1]
    bin_uppers = bins[1:]

    accuracies = torch.zeros(n_bins)
    confidences = torch.zeros(n_bins)
    bin_counts = torch.zeros(n_bins)
    
    max_output = output.max(dim=1)
    predicted_confidences = max_output.values
    predicted_labels = max_output.indices

    def get_bin(conf):
        for idx, bin_upper in enumerate(bin_uppers):
            if conf < bin_upper:
                return idx
        return n_bins-1
    
    bin_to_idx = {}
    for bin_number in range(n_bins):
        bin_to_idx[bin_number] = []
    
    for idx, pred_conf in enumerate(predicted_confidences):
        bin_to_idx[get_bin(pred_conf)].append(idx)
    
    for bin_number in range(n_bins):
        bin_idxs = bin_to_idx[bin_number]
        bin_counts[bin_number] = len(bin_idxs)
        if len(bin_idxs) > 0:
            correct = (true_labels[bin_idxs] == predicted_labels[bin_idxs]).float()
            accuracies[bin_number] = correct.mean()
            confidences[bin_number] = predicted_confidences[bin_idxs].mean()

    if plot:
        plt.figure(figsize=(12, 5))
        plt.suptitle(name)
    
        # Reliability diagram
        plt.subplot(1, 2, 1)
        plt.plot([0, 1], [0, 1], 'k--')
        plt.plot(confidences, accuracies, marker='o')
        plt.xlabel('Confidence')
        plt.ylabel('Accuracy')
        plt.title('Reliability Diagram')
    
        # Histogram 
        plt.subplot(1, 2, 2)
        plt.bar(bin_lowers, bin_counts, width=(bin_uppers[1] - bin_lowers[1]), align='edge', edgecolor='k')
        plt.xlabel('Confidence')
        plt.ylabel('Number of Samples')
        plt.title('Confidence Histogram')
    
        plt.tight_layout()
        plt.savefig(f"{name}.pdf", format="pdf", bbox_inches="tight")
        plt.show()

    # Compute ECE
    valid_bins = bin_counts > 0  # Only consider non-empty bins
    ece = torch.sum(bin_counts[valid_bins] / bin_counts.sum() * torch.abs(accuracies[valid_bins] - confidences[valid_bins]))
    return ece.item()*100

In [5]:
MODEL_FOLDER = "suzipu_models"

In [9]:


ece_results = {}
for test_edition in [Editions.LU, Editions.ZHANG, Editions.SIKU, Editions.ZHU, Editions.SHANGHAI]:
    ece_results[test_edition] = {}
    print(test_edition)
    for label_type in [LabelType.PITCH_BALANCED, LabelType.SECONDARY_BALANCED]:
        print("    ", label_type)
        ece_results[test_edition][label_type] = {}
        ece_results[test_edition][label_type]["uncalibrated"] = []
        ece_results[test_edition][label_type]["calibrated"] = []

        for trial in range(10):
            print("         ", trial)
            model = CnnModel(num_classes = 11 if "pitch" in label_type else 7)
            total_dataloaders = get_all_dataloaders(total_dataset, transformations=transformations, test_edition=test_edition)
            
            model.load_state_dict(torch.load(f"{MODEL_FOLDER}/suzipu-best-{test_edition}-{label_type}.std"))
            calibration_module_pitch = TemperatureScalingCalibrationModule(model)
            calibration_module_pitch.freeze_base_model()
            calibration_module_pitch.fit(total_dataloaders[label_type]["validation"], start_value=1., lr=0.05, n_epochs=5)
        
            output, labels = predict_data_loader(calibration_module_pitch, total_dataloaders[label_type]["test"], scaled=False)
            ece_results[test_edition][label_type]["uncalibrated"].append(reliability_diagram(output, labels, name="Suzipu Pitch (Uncalibrated)", plot=False))
            output, labels = predict_data_loader(calibration_module_pitch, total_dataloaders[label_type]["test"], scaled=True)
            ece_results[test_edition][label_type]["calibrated"].append(reliability_diagram(output, labels, name="Suzipu Pitch (Calibrated)", plot=False))

lu
     pitch_balanced
          0
          1
          2
          3
          4
          5
          6
          7
          8
          9
     secondary_balanced
          0
          1
          2
          3
          4
          5
          6
          7
          8
          9
zhang
     pitch_balanced
          0
          1
          2
          3
          4
          5
          6
          7
          8
          9
     secondary_balanced
          0
          1
          2
          3
          4
          5
          6
          7
          8
          9
siku
     pitch_balanced
          0
          1
          2
          3
          4
          5
          6
          7
          8
          9
     secondary_balanced
          0
          1
          2
          3
          4
          5
          6
          7
          8
          9
zhu
     pitch_balanced
          0
          1
          2
          3
          4
          5
          6
          7
          8
  

In [10]:
print(ece_results)

{'lu': {'pitch_balanced': {'uncalibrated': [1.013728603720665, 1.013728603720665, 1.013728603720665, 1.013728603720665, 1.013728603720665, 1.013728603720665, 1.013728603720665, 1.013728603720665, 1.013728603720665, 1.013728603720665], 'calibrated': [1.166769117116928, 1.2913772836327553, 0.6888074334710836, 0.6359653547406197, 0.9236518293619156, 1.1488262563943863, 1.1372829787433147, 1.4458037912845612, 1.2960443273186684, 0.6978386081755161]}, 'secondary_balanced': {'uncalibrated': [0.39160759188234806, 0.39160759188234806, 0.39160759188234806, 0.39160759188234806, 0.39160759188234806, 0.39160759188234806, 0.39160759188234806, 0.39160759188234806, 0.39160759188234806, 0.39160759188234806], 'calibrated': [0.33447700552642345, 0.6717104464769363, 0.19103933591395617, 0.39329868741333485, 0.2732322784140706, 0.4747429396957159, 0.6478836759924889, 0.22352091036736965, 0.40007042698562145, 0.3180652391165495]}}, 'zhang': {'pitch_balanced': {'uncalibrated': [0.6221192423254251, 0.6221192

In [11]:
aggregated_lists = {}
by_editions_lists = {}
for cali in ["uncalibrated", "calibrated"]:
    aggregated_lists[cali] = {}
    by_editions_lists[cali] = {}
    for label_type in [LabelType.PITCH_BALANCED, LabelType.SECONDARY_BALANCED]:
        aggregated_lists[cali][label_type] = []
        by_editions_lists[cali][label_type] = {}
        for test_edition in [Editions.LU, Editions.ZHANG, Editions.SIKU, Editions.ZHU, Editions.SHANGHAI]:
            by_editions_lists[cali][label_type][test_edition] = []
            for trial in range(10):
                aggregated_lists[cali][label_type].append(ece_results[test_edition][label_type][cali][trial])
                by_editions_lists[cali][label_type][test_edition].append(ece_results[test_edition][label_type][cali][trial])
            by_editions_lists[cali][label_type][test_edition] = np.array(by_editions_lists[cali][label_type][test_edition])/100
        aggregated_lists[cali][label_type] = np.array(aggregated_lists[cali][label_type])/100
            
    
print(f'all uncalibrated  pitch          {aggregated_lists["uncalibrated"]["pitch_balanced"].mean():.4f} +- {aggregated_lists["uncalibrated"]["pitch_balanced"].std():.4f}')
print(f'all calibrated    pitch          {aggregated_lists["calibrated"]["pitch_balanced"].mean():.4f} +- {aggregated_lists["calibrated"]["pitch_balanced"].std():.4f}')
print()
print(f'all uncalibrated  secondary      {aggregated_lists["uncalibrated"]["secondary_balanced"].mean():.4f} +- {aggregated_lists["uncalibrated"]["secondary_balanced"].std():.4f}')
print(f'all calibrated    secondary      {aggregated_lists["calibrated"]["secondary_balanced"].mean():.4f} +- {aggregated_lists["calibrated"]["secondary_balanced"].std():.4f}')
print()
for test_edition in [Editions.LU, Editions.ZHANG, Editions.SIKU, Editions.ZHU, Editions.SHANGHAI]:
    print(f'{test_edition} pitch uncalibrated: {by_editions_lists["uncalibrated"]["pitch_balanced"][test_edition].mean():.4f} +- {by_editions_lists["uncalibrated"]["pitch_balanced"][test_edition].std():.4f}')
    print(f'{test_edition} pitch calibrated:   {by_editions_lists["calibrated"]["pitch_balanced"][test_edition].mean():.4f} +- {by_editions_lists["calibrated"]["pitch_balanced"][test_edition].std():.4f}')
    print()
    print(f'{test_edition} secondary uncalibrated: {by_editions_lists["uncalibrated"]["secondary_balanced"][test_edition].mean():.4f} +- {by_editions_lists["uncalibrated"]["secondary_balanced"][test_edition].std():.4f}')
    print(f'{test_edition} secondary calibrated:   {by_editions_lists["calibrated"]["secondary_balanced"][test_edition].mean():.4f} +- {by_editions_lists["calibrated"]["secondary_balanced"][test_edition].std():.4f}')
    print()
    print()


all uncalibrated  pitch          0.0090 +- 0.0016
all calibrated    pitch          0.0094 +- 0.0026

all uncalibrated  secondary      0.0090 +- 0.0049
all calibrated    secondary      0.0096 +- 0.0045

lu pitch uncalibrated: 0.0101 +- 0.0000
lu pitch calibrated:   0.0104 +- 0.0027

lu secondary uncalibrated: 0.0039 +- 0.0000
lu secondary calibrated:   0.0039 +- 0.0016


zhang pitch uncalibrated: 0.0062 +- 0.0000
zhang pitch calibrated:   0.0070 +- 0.0017

zhang secondary uncalibrated: 0.0039 +- 0.0000
zhang secondary calibrated:   0.0088 +- 0.0024


siku pitch uncalibrated: 0.0089 +- 0.0000
siku pitch calibrated:   0.0092 +- 0.0013

siku secondary uncalibrated: 0.0079 +- 0.0000
siku secondary calibrated:   0.0091 +- 0.0011


zhu pitch uncalibrated: 0.0088 +- 0.0000
zhu pitch calibrated:   0.0103 +- 0.0029

zhu secondary uncalibrated: 0.0154 +- 0.0000
zhu secondary calibrated:   0.0166 +- 0.0006


shanghai pitch uncalibrated: 0.0111 +- 0.0000
shanghai pitch calibrated:   0.0103 +- 0.002

# Save models that performs best on whole KuiSCIMA

In [8]:
def loss_function(x, y, label_type, gamma):
    return FocalLoss(gamma=gamma)(x, y)
    
def test(model, test_loader, LABEL_TYPE, gamma=0):  
    model.eval()
        
    # Train the model
    total_step = len(test_loader)
    
    print(f"Testing with")
    print(LABEL_TYPE)
    
    pred_y = []
    full_labels = []
    is_simple_full = []
        
    for i, (images, labels, is_simple) in enumerate(test_loader):

        # gives batch data, normalize x when iterate train_loader
        b_x = Variable(images)   # batch x
        b_y = Variable(labels)

        output = model(b_x)

        loss = loss_function(output, b_y, LABEL_TYPE, gamma)
        pred_y += torch.argmax(output, dim=1)
        is_simple_full += is_simple
        full_labels += labels
    
    labels = torch.Tensor(full_labels)
    pred_y = torch.Tensor(pred_y)

    is_simple = torch.Tensor(is_simple_full)
    
    total_accuracy = (pred_y == labels).sum() / float(len(pred_y))
    pred_y_simple = torch.Tensor([pred_y[idx] for idx in range(len(pred_y)) if is_simple[idx]])
    labels_simple = torch.Tensor([labels[idx] for idx in range(len(pred_y)) if is_simple[idx]])
    simple_accuracy = (pred_y_simple == labels_simple).sum() / len(pred_y_simple)   

    pred_y_composite = torch.Tensor([pred_y[idx] for idx in range(len(pred_y)) if not is_simple[idx]])
    labels_composite = torch.Tensor([labels[idx] for idx in range(len(pred_y)) if not is_simple[idx]])
    composite_accuracy = (pred_y_composite == labels_composite).sum() / len(pred_y_composite) 

    print ('    Acc.(Total): {:.2f}%, Acc.(Simple): {:.2f}%, Acc.(Composite): {:.2f}%'.format(total_accuracy*100, simple_accuracy*100, composite_accuracy*100))
    
    return total_accuracy


best_models = {}
all_accuracies = {}
for label_type in [LabelType.PITCH_BALANCED, LabelType.SECONDARY_BALANCED]:
    total_dataloaders = get_all_dataloaders(total_dataset, transformations=transformations, test_edition=Editions.SHANGHAI)
    complete_dataset = total_dataloaders[label_type]["train"].dataset + \
                       total_dataloaders[label_type]["validation"].dataset + \
                       total_dataloaders[label_type]["test"].dataset
    complete_dataloader = torch.utils.data.DataLoader(complete_dataset, batch_size=100, shuffle=False)
    best_accuracy = 0
    all_accuracies[label_type] = {}
    for test_edition in [Editions.LU, Editions.ZHANG, Editions.SIKU, Editions.ZHU, Editions.SHANGHAI]:
        model = CnnModel(num_classes = 11 if "pitch" in label_type else 7)
        model.load_state_dict(torch.load(f"{MODEL_FOLDER}/suzipu-best-{test_edition}-{label_type}.std"))
        accuracy = test(model, complete_dataloader, label_type)
        all_accuracies[label_type][test_edition] = accuracy
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_models[label_type] = {"edition": test_edition, "accuracy": accuracy}

print(all_accuracies)
print(best_models)

Testing with
pitch_balanced


/home/tristan/Desktop/git/SuziAI/SuziOMR/baseline_new/model.py:131: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  logpt = F.log_softmax(input)


    Acc.(Total): 98.60%, Acc.(Simple): 99.57%, Acc.(Composite): 96.10%
Testing with
pitch_balanced
    Acc.(Total): 98.76%, Acc.(Simple): 99.42%, Acc.(Composite): 97.05%
Testing with
pitch_balanced
    Acc.(Total): 98.44%, Acc.(Simple): 99.54%, Acc.(Composite): 95.60%
Testing with
pitch_balanced
    Acc.(Total): 98.31%, Acc.(Simple): 99.48%, Acc.(Composite): 95.30%
Testing with
pitch_balanced
    Acc.(Total): 98.51%, Acc.(Simple): 99.38%, Acc.(Composite): 96.25%
Testing with
secondary_balanced
    Acc.(Total): 98.90%, Acc.(Simple): 99.52%, Acc.(Composite): 97.30%
Testing with
secondary_balanced
    Acc.(Total): 98.81%, Acc.(Simple): 99.42%, Acc.(Composite): 97.25%
Testing with
secondary_balanced
    Acc.(Total): 98.69%, Acc.(Simple): 99.40%, Acc.(Composite): 96.85%
Testing with
secondary_balanced
    Acc.(Total): 98.21%, Acc.(Simple): 98.95%, Acc.(Composite): 96.30%
Testing with
secondary_balanced
    Acc.(Total): 98.56%, Acc.(Simple): 99.26%, Acc.(Composite): 96.75%
{'pitch_balanced':

In [135]:
# Save models to file
pitch_model_base = CnnModel(num_classes = 11)
pitch_model_base.load_state_dict(torch.load(f"{MODEL_FOLDER}/suzipu-best-{best_models[LabelType.PITCH_BALANCED]['edition']}-{LabelType.PITCH_BALANCED}.std"))
secondary_model_base = CnnModel(num_classes = 7)
secondary_model_base.load_state_dict(torch.load(f"{MODEL_FOLDER}/suzipu-best-{best_models[LabelType.SECONDARY_BALANCED]['edition']}-{LabelType.SECONDARY_BALANCED}.std"))

pitch_model = TemperatureScalingCalibrationModule(pitch_model_base)
torch.save(pitch_model.state_dict(), f"suzi_model_pitch.std")
secondary_model = TemperatureScalingCalibrationModule(secondary_model_base)
torch.save(secondary_model.state_dict(), f"suzi_model_secondary.std")

## Load models

In [6]:
pitch_model = TemperatureScalingCalibrationModule(CnnModel(num_classes = 11))
pitch_model.load_state_dict(torch.load(f"suzi_model_pitch.std"))
secondary_model = TemperatureScalingCalibrationModule(CnnModel(num_classes = 7))
secondary_model.load_state_dict(torch.load(f"suzi_model_secondary.std"))

total_dataloaders = get_all_dataloaders(total_dataset, transformations=transformations, test_edition=best_models[LabelType.PITCH_BALANCED]['edition'])
output, labels = predict_data_loader(pitch_model, total_dataloaders[LabelType.PITCH_BALANCED]["test"], scaled=False)
print(reliability_diagram(output, labels, name="Suzipu Pitch (Zhang, Uncalibrated)", plot=True)/100)

total_dataloaders = get_all_dataloaders(total_dataset, transformations=transformations, test_edition=best_models[LabelType.SECONDARY_BALANCED]['edition'])
output, labels = predict_data_loader(secondary_model, total_dataloaders[LabelType.SECONDARY_BALANCED]["test"], scaled=False)
print(reliability_diagram(output, labels, name="Suzipu Secondary (Lu, Uncalibrated)", plot=True)/100)


NameError: name 'best_models' is not defined